# Game 1 - The Mirror Maze

**Team:** Ded_Sec

This completed notebook verifies the generated evidence and reproduces the
provided text baseline before and after the corrected group-aware split.


In [1]:
from pathlib import Path
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

ROOT = Path(os.environ.get("AIO_ROOT", r"C:\Users\mohd1\OneDrive\Desktop\ai olpyc\AI_Olympics_2026_Student_Release_v1"))
GAME1_DIR = ROOT / "games" / "game1_mirror_maze"
OUTPUT_DIR = Path.cwd()

naive_train = pd.read_csv(GAME1_DIR / "game1_train_naive.csv")
naive_validation = pd.read_csv(GAME1_DIR / "game1_validation_naive.csv")
corrected_train = pd.read_csv(OUTPUT_DIR / "game1_corrected_train.csv")
corrected_validation = pd.read_csv(OUTPUT_DIR / "game1_corrected_validation.csv")
evidence = pd.read_csv(OUTPUT_DIR / "game1_evidence_table.csv")

print("Naive:", len(naive_train), len(naive_validation))
print("Corrected:", len(corrected_train), len(corrected_validation))
print("Evidence rows:", len(evidence))


Naive: 10000 2500
Corrected: 9999 2501
Evidence rows: 1555


In [2]:
def baseline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2,
            max_features=40000,
            sublinear_tf=True,
        )),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ])

def evaluate(train_df, validation_df):
    model = baseline()
    model.fit(train_df["text"].fillna(""), train_df["label"])
    predictions = model.predict(validation_df["text"].fillna(""))
    return {
        "accuracy": accuracy_score(validation_df["label"], predictions),
        "macro_f1": f1_score(
            validation_df["label"], predictions, average="macro"
        ),
        "report": classification_report(
            validation_df["label"], predictions, digits=4
        ),
        "confusion_matrix": confusion_matrix(
            validation_df["label"], predictions
        ),
    }


In [3]:
initial = evaluate(naive_train, naive_validation)
corrected = evaluate(corrected_train, corrected_validation)

comparison = pd.DataFrame([
    {
        "evaluation_setup": "Initial validation",
        "accuracy": initial["accuracy"],
        "macro_f1": initial["macro_f1"],
    },
    {
        "evaluation_setup": "Corrected validation",
        "accuracy": corrected["accuracy"],
        "macro_f1": corrected["macro_f1"],
    },
])
display(comparison)
print("\nInitial classification report:\n", initial["report"])
print("Initial confusion matrix:\n", initial["confusion_matrix"])
print("\nCorrected classification report:\n", corrected["report"])
print("Corrected confusion matrix:\n", corrected["confusion_matrix"])


,evaluation_setup,accuracy,macro_f1
0,Initial validation,0.790800,0.787967
1,Corrected validation,0.762095,0.759162



Initial classification report:
               precision    recall  f1-score   support

        fake     0.8783    0.6752    0.7635      1250
        real     0.7362    0.9064    0.8125      1250

    accuracy                         0.7908      2500
   macro avg     0.8072    0.7908    0.7880      2500
weighted avg     0.8072    0.7908    0.7880      2500

Initial confusion matrix:
 [[ 844  406]
 [ 117 1133]]

Corrected classification report:
               precision    recall  f1-score   support

        fake     0.8368    0.6515    0.7326      1251
        real     0.7145    0.8728    0.7857      1250

    accuracy                         0.7621      2501
   macro avg     0.7756    0.7621    0.7592      2501
weighted avg     0.7756    0.7621    0.7592      2501

Corrected confusion matrix:
 [[ 815  436]
 [ 159 1091]]


In [4]:
display(
    evidence.groupby("evidence_type")
    .size()
    .rename("cases")
    .sort_values(ascending=False)
    .to_frame()
)
display(evidence.head(20))


,cases
evidence_type,
perceptual_image_hash_distance,602
exact_normalized_text,600
near_text_cosine,353


,validation_sample_id,related_train_sample_id,evidence_type,similarity_score_or_distance,label
0,mirror_leak__0000,train__006756,exact_normalized_text,1.0,real
1,mirror_leak__0000,train__006756,perceptual_image_hash_distance,0.0,real
2,mirror_leak__0001,train__002938,exact_normalized_text,1.0,fake
3,mirror_leak__0001,train__002938,near_text_cosine,1.0,fake
4,mirror_leak__0001,train__002938,perceptual_image_hash_distance,0.0,fake
5,mirror_leak__0002,train__002173,exact_normalized_text,1.0,fake
6,mirror_leak__0002,train__002173,near_text_cosine,1.0,fake
7,mirror_leak__0002,train__002173,perceptual_image_hash_distance,0.0,fake
8,mirror_leak__0003,train__008500,exact_normalized_text,1.0,real
9,mirror_leak__0003,train__008500,near_text_cosine,1.0,real


## Final Conclusion

The initial validation score was not trustworthy. The audit found 1,555 train-validation evidence rows caused by normalized text overlap and exact or perceptually similar images. Related samples were connected into groups and assigned to one fold, leaving 0 known relation edges across the corrected boundary. The unchanged TF-IDF plus logistic-regression baseline changed from accuracy 0.7908 and macro-F1 0.7880 to accuracy 0.7621 and macro-F1 0.7592. This demonstrates why leakage auditing must precede stronger model training.
